In [2]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time
from functools import partial

In [3]:
import random

In [4]:
random.seed(42)

In [5]:
tests = ['gc_50_3', 'gc_70_7', 'gc_100_5', 'gc_250_9', 'gc_500_1', 'gc_1000_5']
thresholds = [(8, 6), (20, 17), (21, 16), (95, 78), (18, 16), (124, 100)]

In [6]:
def load_graph(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n, m = list(map(int, lines[0].split()))
        edges = list()
        for i in range(m):
            u, v = list(map(int, lines[1 + i].split()))
            edges.append((u, v))

        return n, m, edges

In [7]:
def check_coloring(n, m, edges, color):
    if min(color) <= 0:
        raise Exception("Not correct coloring")
        
    for i in range(m):
        u, v = edges[i]
        if color[u] == color[v]:
            raise Exception("Not correct coloring")
                
    result = max(color)
    return result

In [8]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [9]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, m, edges = load_graph(test)
        start = time.time()
        
        if not use_file:
            coloring = method(n, m, edges)
        else:
            coloring = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_coloring(n, m, edges, coloring)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Сначала напишем жадный алгоритм, который проходится по вершинам в некотором порядке и красит вершину в минимальный возможный цвет. 
Будем пробовать несколько случайных порядков и выберем лучший.

In [10]:
!g++ -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [11]:
def greedy_coloring(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        coloring = list(map(int, lines[0].split()))
        return coloring

In [12]:
test_method(greedy_coloring, "greedy", True)

Checking greedy
Execution time: 0.4824 seconds
Target function gc_50_3: 7
Passed gc_50_3: 1
Execution time: 0.2412 seconds
Target function gc_70_7: 20
Passed gc_70_7: 1
Execution time: 0.3419 seconds
Target function gc_100_5: 19
Passed gc_100_5: 1
Execution time: 4.3285 seconds
Target function gc_250_9: 94
Passed gc_250_9: 1
Execution time: 1.5519 seconds
Target function gc_500_1: 18
Passed gc_500_1: 1
Execution time: 32.0386 seconds
Target function gc_1000_5: 123
Passed gc_1000_5: 1
Score: 18


Далее рассмотрим метод имитации отжига. На каждой итерации будем случайным образом выбирать две вершины и менять их местами, после чего запускать жадный алгоритм для построения решения. Также будем использовать рестарты, чтобы уменьшить вероятность застревания в неудачном локальном минимуме.

Интуитивно, для графов большего размера может требоваться больше итераций для нахождения хорошего решения. Поэтому количество рестартов будем выбирать в зависимости от размера графа.

В качестве гиперпараметра будем перебирать коэффициент, во сколько раз температура уменьшается на каждой итерации.

In [13]:
!g++ -std=c++2a cpp_methods/sa.cpp -o tmp/sa

In [14]:
def sa_coloring(test_file, step_temp=0.9): 
    os.system(f"./tmp/sa data/{test_file} {step_temp}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        coloring = list(map(int, lines[0].split()))
        return coloring

In [15]:
test_method(partial(sa_coloring, step_temp=0.9), "sa", True)

Checking sa
Execution time: 37.5289 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 33.1207 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 32.7796 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 29.7446 seconds
Target function gc_250_9: 85
Passed gc_250_9: 1
Execution time: 32.9639 seconds
Target function gc_500_1: 17
Passed gc_500_1: 1
Execution time: 30.0583 seconds
Target function gc_1000_5: 121
Passed gc_1000_5: 1
Score: 24


In [16]:
test_method(partial(sa_coloring, step_temp=0.99), "sa", True)

Checking sa
Execution time: 37.4162 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 32.8568 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 32.9924 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 29.6868 seconds
Target function gc_250_9: 85
Passed gc_250_9: 1
Execution time: 32.8602 seconds
Target function gc_500_1: 18
Passed gc_500_1: 1
Execution time: 30.0025 seconds
Target function gc_1000_5: 120
Passed gc_1000_5: 1
Score: 24


In [17]:
test_method(partial(sa_coloring, step_temp=0.999), "sa", True)

Checking sa
Execution time: 37.1587 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 32.5843 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 33.1712 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 29.5269 seconds
Target function gc_250_9: 86
Passed gc_250_9: 1
Execution time: 32.6001 seconds
Target function gc_500_1: 18
Passed gc_500_1: 1
Execution time: 30.1458 seconds
Target function gc_1000_5: 120
Passed gc_1000_5: 1
Score: 24


In [18]:
test_method(partial(sa_coloring, step_temp=0.9999), "sa", True)

Checking sa
Execution time: 37.5022 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 32.8762 seconds
Target function gc_70_7: 19
Passed gc_70_7: 1
Execution time: 33.0401 seconds
Target function gc_100_5: 18
Passed gc_100_5: 1
Execution time: 29.6495 seconds
Target function gc_250_9: 92
Passed gc_250_9: 1
Execution time: 32.6471 seconds
Target function gc_500_1: 18
Passed gc_500_1: 1
Execution time: 29.8389 seconds
Target function gc_1000_5: 120
Passed gc_1000_5: 1
Score: 20


Изменение на уровне 0.99 оказалось наилучшим с 24 баллами.

Сделаем существенное изменение метода, а именно модифицируем объект, который отжиг будет менять.

Замена двух случайных вершин выглядит не самым естественным изменением для метода имитации отжига.

Заметим следующее свойство: если упорядочить вершины по номерам их цветов, то число цветов в решении не увеличится. Действительно, вершины одного цвета образуют независимое множество, поэтому вместо перестановки отдельных вершин можно переставлять целые блоки вершин, соответствующие одному цвету.

Таким образом, в качестве соседнего состояния для отжига будем рассматривать перестановку блоков цветов.

Сначала был протестирован вариант, в котором на каждой итерации менялись местами два случайных цветовых блока. Однако на практике он показал себя хуже, чем операция реверса случайного подотрезка блоков цветов. Я буду использовать именно такое изменения и также перебирать изменение температуры в отжиге.

In [21]:
!g++ -std=c++2a cpp_methods/tuned_sa.cpp -o tmp/tuned_sa

In [22]:
def tuned_sa_coloring(test_file, step_temp=0.99): 
    os.system(f"./tmp/tuned_sa data/{test_file} {step_temp}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        coloring = list(map(int, lines[0].split()))
        return coloring

In [23]:
test_method(partial(tuned_sa_coloring, step_temp=0.9), "tuned_sa", True)

Checking tuned_sa
Execution time: 66.2776 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 52.0009 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 52.3557 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 46.5291 seconds
Target function gc_250_9: 78
Passed gc_250_9: 2
Execution time: 51.5379 seconds
Target function gc_500_1: 14
Passed gc_500_1: 2
Execution time: 47.0366 seconds
Target function gc_1000_5: 106
Passed gc_1000_5: 1
Score: 28


In [25]:
test_method(partial(tuned_sa_coloring, step_temp=0.99), "tuned_sa", True)

Checking tuned_sa
Execution time: 65.8526 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 51.9987 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 56.5909 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 52.6804 seconds
Target function gc_250_9: 78
Passed gc_250_9: 2
Execution time: 52.7887 seconds
Target function gc_500_1: 14
Passed gc_500_1: 2
Execution time: 49.4715 seconds
Target function gc_1000_5: 106
Passed gc_1000_5: 1
Score: 28


In [26]:
test_method(partial(tuned_sa_coloring, step_temp=0.999), "tuned_sa", True)

Checking tuned_sa
Execution time: 74.3240 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 58.8901 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 59.2513 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 52.5222 seconds
Target function gc_250_9: 78
Passed gc_250_9: 2
Execution time: 58.0994 seconds
Target function gc_500_1: 14
Passed gc_500_1: 2
Execution time: 53.1281 seconds
Target function gc_1000_5: 106
Passed gc_1000_5: 1
Score: 28


In [27]:
test_method(partial(tuned_sa_coloring, step_temp=0.5), "tuned_sa", True)

Checking tuned_sa
Execution time: 75.1627 seconds
Target function gc_50_3: 6
Passed gc_50_3: 2
Execution time: 59.2548 seconds
Target function gc_70_7: 17
Passed gc_70_7: 2
Execution time: 59.6957 seconds
Target function gc_100_5: 16
Passed gc_100_5: 2
Execution time: 52.6178 seconds
Target function gc_250_9: 78
Passed gc_250_9: 2
Execution time: 58.8974 seconds
Target function gc_500_1: 14
Passed gc_500_1: 2
Execution time: 53.4035 seconds
Target function gc_1000_5: 106
Passed gc_1000_5: 1
Score: 28


Прошли еще +2 сложных порога! От подбора гиперпараметров фактически ничего не меняется.